# Python for Applied AI — Day 2
## Advanced Python: Decorators, Iterators, Generators & Context Managers

**Duration:** 6 hours
**Level:** 2 (Intermediate)

### Today's Agenda
1. Iterators — the iteration protocol
2. Generators and `yield`
3. Generator expressions
4. Decorators — functions that modify functions
5. Practical decorator patterns
6. Context Managers — `with` statement
7. Building a custom Context Manager
8. `contextlib` shortcuts
9. Practice exercises

> **Note:** This session introduces the concepts and includes light hands-on coding, without covering every detail exhaustively (as agreed for Level 2).


## 1. Iterators — The Iteration Protocol

An **iterable** is any object you can loop over (`list`, `str`, `dict`, etc.). An **iterator** is the object that actually produces the next value, one at a time.

An object is an iterator if it implements:
- `__iter__()` — returns the iterator object itself
- `__next__()` — returns the next value, or raises `StopIteration` when exhausted

Understanding this protocol is the foundation for understanding generators.


In [1]:
numbers = [1, 2, 3]
iterator = iter(numbers)   # calls numbers.__iter__()

print(next(iterator))  # calls iterator.__next__()
print(next(iterator))
print(next(iterator))

try:
    print(next(iterator))
except StopIteration:
    print("No more items!")


1
2
3
No more items!


In [2]:
class CountUpTo:
    """A custom iterator that counts from 1 up to a limit."""
    def __init__(self, limit):
        self.limit = limit
        self.current = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < self.limit:
            self.current += 1
            return self.current
        raise StopIteration

counter = CountUpTo(5)
for num in counter:
    print(num)


1
2
3
4
5


### Extra Example: Iterator Over a Custom Collection

In [3]:
class ReverseIterator:
    """Iterates over a sequence in reverse order."""
    def __init__(self, data):
        self.data = data
        self.index = len(data)

    def __iter__(self):
        return self

    def __next__(self):
        if self.index == 0:
            raise StopIteration
        self.index -= 1
        return self.data[self.index]

for letter in ReverseIterator("PYTHON"):
    print(letter)


N
O
H
T
Y
P


## 2. Generators and `yield`

Writing a full iterator class (like above) is verbose. **Generators** let you write iterators using regular functions with the `yield` keyword.

Key idea: each time `yield` is hit, the function's state is **paused**, and resumed the next time `next()` is called. This makes generators **memory-efficient** — values are produced lazily, one at a time, instead of building a whole list in memory.


In [4]:
def count_up_to(limit):
    current = 1
    while current <= limit:
        yield current
        current += 1

gen = count_up_to(5)
print(type(gen))

for num in gen:
    print(num)


<class 'generator'>
1
2
3
4
5


In [5]:
# Memory efficiency demonstration
def infinite_counter():
    n = 1
    while True:
        yield n
        n += 1

counter = infinite_counter()
for _ in range(5):
    print(next(counter))
# The generator never stores all numbers in memory — only the current state


1
2
3
4
5


### Extra Example: A Practical Generator — Reading Large Data in Chunks

In [6]:
def batch_generator(data, batch_size):
    """Yields successive batches from a list — common pattern when feeding data to an AI model."""
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

dataset = list(range(1, 23))  # imagine this is 1 million training samples
for batch in batch_generator(dataset, batch_size=5):
    print("Processing batch:", batch)


Processing batch: [1, 2, 3, 4, 5]
Processing batch: [6, 7, 8, 9, 10]
Processing batch: [11, 12, 13, 14, 15]
Processing batch: [16, 17, 18, 19, 20]
Processing batch: [21, 22]


## 3. Generator Expressions

Just like list comprehensions, but with `()` instead of `[]`. They produce values lazily instead of building the full list in memory — useful for large datasets, common in AI/data pipelines.


In [7]:
# List comprehension: builds the entire list in memory immediately
squares_list = [x**2 for x in range(1000000)]

# Generator expression: produces values one at a time, lazily
squares_gen = (x**2 for x in range(1000000))

print(type(squares_list), type(squares_gen))
print(next(squares_gen))
print(next(squares_gen))


<class 'list'> <class 'generator'>
0
1


### Extra Example: Generator with a `return` Value and `yield from`

In [8]:
def numbers_up_to(n):
    for i in range(1, n + 1):
        yield i
    return "done"

def combined_generator():
    yield from numbers_up_to(3)   # delegates to another generator
    yield from numbers_up_to(2)

for val in combined_generator():
    print(val)

# Capturing the return value of a generator
gen = numbers_up_to(3)
try:
    while True:
        print("value:", next(gen))
except StopIteration as e:
    print("Generator finished with return value:", e.value)


1
2
3
1
2
value: 1
value: 2
value: 3
Generator finished with return value: done


## 4. Decorators — Functions That Modify Functions

A **decorator** is a function that takes another function as input and extends its behavior without permanently modifying it. This relies on the fact that in Python, functions are **first-class objects** — they can be passed around like any other variable.

The `@decorator_name` syntax above a function is just syntactic sugar for `function = decorator_name(function)`.


In [9]:
def my_decorator(func):
    def wrapper():
        print("Something happens before the function runs.")
        func()
        print("Something happens after the function runs.")
    return wrapper

def say_hello():
    print("Hello!")

say_hello = my_decorator(say_hello)  # manual decoration
say_hello()


Something happens before the function runs.
Hello!
Something happens after the function runs.


In [10]:
# Same thing using the @ syntax
def my_decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()


Before
Hello!
After


### Decorators with Arguments

To decorate functions that accept arguments, the wrapper must accept `*args, **kwargs` and pass them through.


In [11]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def slow_add(a, b):
    time.sleep(1)
    return a + b

result = slow_add(3, 4)
print("Result:", result)


slow_add took 1.0002 seconds
Result: 7


### Extra Example: Stacking Multiple Decorators

In [12]:
def uppercase(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

def exclaim(func):
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result + "!!!"
    return wrapper

@exclaim
@uppercase
def greet(name):
    return f"hello, {name}"

# Decorators apply bottom-up: greet -> uppercase -> exclaim
print(greet("ahmed"))


HELLO, AHMED!!!


## 5. Practical Decorator Patterns

Decorators are widely used in real-world Python and AI applications for:
- **Logging** — recording function calls and results
- **Timing / performance monitoring**
- **Access control / authentication**
- **Caching** (e.g. `functools.lru_cache`)
- **Retry logic** for unreliable operations (e.g. API calls to an LLM)


In [13]:
from functools import wraps

def log_calls(func):
    @wraps(func)  # preserves the original function's name/docstring
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b):
    """Adds two numbers."""
    return a + b

print(add(2, 3))
print(add.__name__)   # stays 'add' thanks to @wraps


Calling add with args=(2, 3), kwargs={}
5
add


In [14]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

print(fibonacci(30))  # fast, thanks to caching


832040


### Extra Example: A Decorator with Arguments (Decorator Factory)

In [15]:
def repeat(times):
    """A decorator FACTORY — a function that returns a decorator."""
    def decorator(func):
        def wrapper(*args, **kwargs):
            results = []
            for _ in range(times):
                results.append(func(*args, **kwargs))
            return results
        return wrapper
    return decorator

@repeat(times=3)
def roll_dice():
    import random
    return random.randint(1, 6)

print(roll_dice())


[3, 1, 3]


## 6. Context Managers — The `with` Statement

A context manager handles **setup and teardown** logic automatically — most commonly seen with file handling:

```python
with open("file.txt") as f:
    data = f.read()
# file is automatically closed here, even if an error occurred
```

This is safer than manually calling `.close()`, because cleanup happens even if an exception is raised inside the block.


In [16]:
with open("/tmp/demo.txt", "w") as f:
    f.write("Hello, Context Manager!")

with open("/tmp/demo.txt", "r") as f:
    content = f.read()

print(content)


Hello, Context Manager!


## 7. Building a Custom Context Manager

A context manager is any object implementing:
- `__enter__(self)` — setup logic; return value is bound to the `as` variable
- `__exit__(self, exc_type, exc_value, traceback)` — cleanup logic, always runs

We won't cover every detail of exception propagation here — just enough to understand the pattern.


In [17]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.time()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        import time
        elapsed = time.time() - self.start
        print(f"Elapsed time: {elapsed:.4f} seconds")

with Timer():
    total = sum(range(1000000))
    print("Sum computed:", total)


Sum computed: 499999500000
Elapsed time: 0.0134 seconds


### Extra Example: Context Manager That Manages a Resource and Handles Exceptions

In [18]:
class ManagedResource:
    def __enter__(self):
        print("Resource acquired.")
        return self

    def do_something_risky(self):
        raise ValueError("Something went wrong inside the block!")

    def __exit__(self, exc_type, exc_value, traceback):
        print("Cleaning up resource...")
        if exc_type is not None:
            print(f"Handled an exception: {exc_value}")
        return True  # suppresses the exception from propagating further

with ManagedResource() as res:
    res.do_something_risky()

print("Program continues normally after the 'with' block.")


Resource acquired.
Cleaning up resource...
Handled an exception: Something went wrong inside the block!
Program continues normally after the 'with' block.


## 8. `contextlib` Shortcuts (Quick Look)

The `contextlib` module lets you create a context manager from a generator function using `@contextmanager`, avoiding the need to write a full class.


In [19]:
from contextlib import contextmanager
import time

@contextmanager
def timer():
    start = time.time()
    yield              # code inside the 'with' block runs here
    elapsed = time.time() - start
    print(f"Elapsed: {elapsed:.4f} seconds")

with timer():
    total = sum(range(1000000))
    print("Sum:", total)


Sum: 499999500000
Elapsed: 0.0134 seconds


### Extra Example: `contextlib` — Temporarily Changing State

In [20]:
from contextlib import contextmanager

@contextmanager
def temporary_setting(config, key, temp_value):
    original_value = config.get(key)
    config[key] = temp_value
    try:
        yield config
    finally:
        config[key] = original_value  # always restored, even on error

settings = {"debug": False}

with temporary_setting(settings, "debug", True):
    print("Inside block, debug =", settings["debug"])

print("Outside block, debug =", settings["debug"])


Inside block, debug = True
Outside block, debug = False


## 9. Practice Exercises

1. Write a generator function `even_numbers(limit)` that yields even numbers from 0 up to `limit`.
2. Write a decorator `retry(func)` that retries a function up to 3 times if it raises an exception.
3. Write a custom context manager class `SuppressErrors` that catches and ignores any exception raised inside the `with` block.
4. Rewrite exercise 3 using `@contextmanager` from `contextlib` instead of a class.


In [21]:
# Space for exercise solutions
# Exercise 1
def even_numbers(limit):
    for n in range(0, limit + 1, 2):
        yield n

for n in even_numbers(10):
    print(n)


0
2
4
6
8
10


---
### Summary
Today we covered how Python's iteration protocol underlies **generators**, how **decorators** let us extend function behavior cleanly, and how **context managers** ensure reliable setup/teardown logic. These are essential "Advanced Python" building blocks used throughout real-world AI and backend engineering code.
